In [1]:
%%capture
# LLM chay TU XA tren Modal GPU => notebook chi can client + retrieval (khong cai vLLM/torch nang o day).
# Chay 1 lan, KHONG can restart kernel -> Run All chay mot mach.
%uv pip install -q qdrant-client sentence-transformers 'transformers==5.14.1' 'vllm==0.25.1' python-dotenv pandas scikit-learn tqdm numpy


In [2]:
import ast
import gc
import hashlib
import json
import platform
import os
import random
import re
import time
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Chay local + offload LLM sang Modal GPU. Can: MODAL_TOKEN_ID/MODAL_TOKEN_SECRET (Modal CLI hoac .env),
# HF_TOKEN (tai model o remote), QDRANT_URL/QDRANT_API_KEY (retrieval chay local).
env_path = find_dotenv(usecwd=True)
if not env_path:
    for candidate in [Path.cwd() / '.env', Path.cwd().parent / '.env', Path('/root/.env'), Path('../.env')]:
        if candidate.exists():
            env_path = str(candidate.resolve())
            break
if env_path:
    load_dotenv(env_path, override=False)
    print('Loaded .env:', env_path)
else:
    print('Khong tim thay .env; doc MODAL_TOKEN_ID/MODAL_TOKEN_SECRET/QDRANT_URL/QDRANT_API_KEY/HF_TOKEN tu environment.')
# Tren Modal Notebooks: KHONG can MODAL_TOKEN_* (kernel da xac thuc san),
# va HF_TOKEN/QDRANT_URL/QDRANT_API_KEY do Modal Secret bom vao environment.
for _k in ['HF_TOKEN', 'QDRANT_URL', 'QDRANT_API_KEY']:
    print(' ', _k, 'OK' if os.environ.get(_k) else 'MISSING -> gan Modal Secret hoac dat bien moi truong')
for _k in ['MODAL_TOKEN_ID', 'MODAL_TOKEN_SECRET']:
    print(' ', _k, 'OK' if os.environ.get(_k) else 'trong (BINH THUONG neu chay ngay tren Modal Notebooks)')

# Goc project (noi co .env) -> dung de tim dataset & .env cho Modal, khong phu thuoc cwd cua kernel.
PROJECT_ROOT = Path(env_path).parent if env_path else Path.cwd()
print('PROJECT_ROOT:', PROJECT_ROOT)

# Mỗi notebook chỉ load một model đầy đủ lên GPU.
AVAILABLE_MODELS = ['Qwen3-4B']
MODEL_REPOS = {
    'Llama-3.1-8B-Instruct': 'meta-llama/Llama-3.1-8B-Instruct',
    'Qwen3-4B': 'Qwen/Qwen3-4B',
    'Qwen2.5-7B-Instruct': 'Qwen/Qwen2.5-7B-Instruct',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    'DeepSeek-R1-Distill-Llama-8B': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',
    'Llama-3.2-3B': 'meta-llama/Llama-3.2-3B-Instruct',
}

# Chỉ decoding profile được phép khác nhau theo khuyến nghị của nhà sản xuất.
# Mọi retrieval, prompt content, token budget, seed và metric ở dưới đều giống nhau.
MODEL_GENERATION_PROFILES = {
    'Llama-3.1-8B-Instruct': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Llama-3.2-3B': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Qwen2.5-7B-Instruct': {
        'profile_name': 'vendor_qwen2_5_instruct',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.7, 'top_p': 0.8,
            'top_k': 20, 'repetition_penalty': 1.05,
        },
    },
    'Qwen3-4B': {
        'profile_name': 'vendor_qwen3_thinking',
        'enable_thinking': True,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95, 'top_k': 20,
        },
    },
    'DeepSeek-R1-Distill-Qwen-1.5B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
    'DeepSeek-R1-Distill-Llama-8B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
}

assert len(AVAILABLE_MODELS) == 1, 'Mỗi lần chỉ load một model đầy đủ lên GPU.'
MODEL_NAME = AVAILABLE_MODELS[0]
MODEL_ID = MODEL_REPOS[MODEL_NAME]
# Exact Qwen3-4B revision đã được version 1 sử dụng.
MODEL_REVISION = os.getenv('MODEL_REVISION', '1cfa9a7208912126459214e8b04321603b3df60c')
MODELS_TO_RUN = [MODEL_NAME]
ACTIVE_PROFILE = MODEL_GENERATION_PROFILES[MODEL_NAME]

BENCHMARK_VERSION = 'v10_legal_pado_thinking_base_matched'
BENCHMARK_PROTOCOL = 'legal_pado_thinking_mapper_3vote_derived_verdict'
BGE_MODEL_ID = 'BAAI/bge-m3'
QDRANT_COLLECTION = 'laws_bge_m3_v2_correct_pooling'
EXPECTED_VECTOR_DIM = 1024
TOP_K = 14
MAX_INPUT_TOKENS = 24000
PADO_PIPELINE_VERSION = 'legal-pado-v10-thinking-base-matched-derived-verdict'
# Thinking mode (khớp base) cần token cho reasoning + JSON; ceiling theo base = 8192.
# (2026-08-10) THEM: model KHONG tat duoc thinking theo tung stage (vd DeepSeek-R1-Distill,
# SUPPORTS_THINKING_TOGGLE=False) van "nghi" o issue_mapper/direct_fallback du prompt yeu
# cau enable_thinking=False -> can gap doi ngan sach de tranh bi cat cut, khop dung co che
# da kiem chung o fullchunks/pado (v12/v15).
SUPPORTS_THINKING_TOGGLE = 'qwen3' in MODEL_ID.lower()
NEEDS_THINKING_BUDGET = bool(ACTIVE_PROFILE['enable_thinking']) and not SUPPORTS_THINKING_TOGGLE
MAX_NEW_TOKENS_ISSUE = 8192 if NEEDS_THINKING_BUDGET else 4096
MAX_NEW_TOKENS_JUDGE = 16384
MAX_NEW_TOKENS_DIRECT = 4096 if NEEDS_THINKING_BUDGET else 2048
JUDGE_VOTES = 3
MAX_NEW_TOKENS = max(MAX_NEW_TOKENS_ISSUE, MAX_NEW_TOKENS_JUDGE, MAX_NEW_TOKENS_DIRECT)

# ---- vLLM engine knobs ----
ENABLE_THINKING = True  # Khớp base (vendor_qwen3_thinking) để so sánh công bằng.
USE_STRUCTURED_OUTPUTS = False  # Base sinh JSON tự do + parse robust; grammar sẽ đổi phân phối decoding.
VLLM_DTYPE = os.getenv('VLLM_DTYPE', 'bfloat16')  # khớp base; dùng L4/A10/A100
assert VLLM_DTYPE == 'bfloat16', 'Base đã chạy BF16; đổi dtype sẽ làm mất tính so sánh.'
GPU_MEM_UTIL = float(os.getenv('VLLM_GPU_MEM_UTIL', '0.92'))
MAX_MODEL_LEN = int(os.getenv('VLLM_MAX_MODEL_LEN', '40960'))
MAX_NUM_SEQS = int(os.getenv('VLLM_MAX_NUM_SEQS', '64'))
VLLM_ENFORCE_EAGER = os.getenv('VLLM_ENFORCE_EAGER', '0') == '1'
VLLM_RUNTIME_VERSION = '0.25.1'
# Qwen3 thinking vendor profile (khớp base); seed riêng cho từng vote để reproducible.
SAMPLING = {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20}
MAX_ARTICLE_CHARS = 6000  # giới hạn theo từng điều; mọi model nhận cùng chuỗi evidence
MAX_GENERATION_ATTEMPTS = 1  # strict: mỗi stage chỉ được gọi đúng một lần
FAIL_FAST = False
INVALID_OUTPUT_LABEL = '__INVALID_OUTPUT__'
EVAL_SEEDS = [2026]
OUTPUT_DIR = Path('outputs_alqac_e2e') / BENCHMARK_VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ['A_WIN', 'B_WIN', 'PARTIAL_A_WIN', 'PARTIAL_B_WIN']
assert MAX_GENERATION_ATTEMPTS == 1
assert len(EVAL_SEEDS) == len(set(EVAL_SEEDS))
random.seed(EVAL_SEEDS[0])
np.random.seed(EVAL_SEEDS[0])

def get_secret(*names, required=True):
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    if required:
        raise RuntimeError(
            f'Thiếu secret {names}. Hãy gắn Modal Secret vào notebook hoặc khai báo biến môi trường tương ứng.'
        )
    return None

HF_TOKEN = get_secret('HF_TOKEN', required=False)
QDRANT_URL = get_secret('QDRANT_URL')
QDRANT_API_KEY = get_secret('QDRANT_API_KEY', 'QDRANT_KEY')

# LLM chay tren Modal GPU (remote). Notebook chi can CPU; BGE-M3 embedding dung GPU neu co, khong thi CPU.
EMBED_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(EVAL_SEEDS[0])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(EVAL_SEEDS[0])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Embed device:', EMBED_DEVICE, '| local torch:', torch.__version__)
print('Model (remote vLLM):', MODEL_NAME, '->', MODEL_ID)
print('Benchmark:', BENCHMARK_VERSION, '| protocol:', BENCHMARK_PROTOCOL)
print('Generation profile:', ACTIVE_PROFILE['profile_name'], '| seeds:', EVAL_SEEDS)
print('vLLM dtype:', VLLM_DTYPE, '| unquantized')


Khong tim thay .env; doc MODAL_TOKEN_ID/MODAL_TOKEN_SECRET/QDRANT_URL/QDRANT_API_KEY/HF_TOKEN tu environment.
  HF_TOKEN OK
  QDRANT_URL OK
  QDRANT_API_KEY OK
  MODAL_TOKEN_ID OK
  MODAL_TOKEN_SECRET OK
PROJECT_ROOT: /root
Embed device: cuda | local torch: 2.11.0+cu130
Model (remote vLLM): Qwen3-4B -> Qwen/Qwen3-4B
Benchmark: v10_legal_pado_thinking_base_matched | protocol: legal_pado_thinking_mapper_3vote_derived_verdict
Generation profile: vendor_qwen3_thinking | seeds: [2026]
vLLM dtype: bfloat16 | unquantized


In [3]:
def find_public_test():
    # Có thể override mà không sửa notebook: ALQAC_PUBLIC_TEST_PATH=/path/to/file.json
    candidates = []
    if os.getenv('ALQAC_PUBLIC_TEST_PATH'):
        candidates.append(Path(os.environ['ALQAC_PUBLIC_TEST_PATH']))
    candidates += [
        PROJECT_ROOT / 'data' / 'ALQAC2026_public_test.json',
        PROJECT_ROOT / 'ALQAC2026_public_test.json',
        Path.cwd() / 'ALQAC2026_public_test.json',
        Path('/root/ALQAC2026_public_test.json'),
        Path('/kaggle/input/datasets/ldhhieu18/demnguoctoibinhminh/ALQAC2026_public_test.json'),
        Path('data/ALQAC2026_public_test.json'),
        Path('../data/ALQAC2026_public_test.json'),
        Path('/kaggle/working/ALQAC2026_public_test.json'),
    ]
    # --- Modal Notebooks: file ban upload thuong nam canh notebook (cwd) hoac trong home/root. ---
    candidates += [
        Path.home() / 'ALQAC2026_public_test.json',
        Path('/root/data/ALQAC2026_public_test.json'),
        Path('/workspace/ALQAC2026_public_test.json'),
        Path('/notebooks/ALQAC2026_public_test.json'),
    ]
    for _root in {Path.cwd(), Path.home(), Path('/root')}:
        try:
            if _root.exists():
                candidates.extend(sorted(_root.rglob('ALQAC2026_public_test.json'))[:5])
        except (PermissionError, OSError):
            pass
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob('ALQAC2026_public_test.json'))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    checked = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(f'Không tìm thấy ALQAC2026_public_test.json. Đã kiểm tra:\n{checked}')


DATA_PATH = find_public_test()
with DATA_PATH.open(encoding='utf-8') as f:
    public_data = json.load(f)

assert len(public_data) == 50, f'Expected 50 cases, got {len(public_data)}'
assert len({x['case_id'] for x in public_data}) == len(public_data)
assert all(x.get('case_query') for x in public_data)
assert all(x.get('verdict_label') in LABELS for x in public_data)

# ---- Nạp CHUNK (agent_v4_results.json) làm case_facts bổ sung ngoài case_query ----
def find_data_file(name):
    candidates = []
    if os.getenv('ALQAC_' + name.upper().replace('.', '_') + '_PATH'):
        candidates.append(Path(os.environ['ALQAC_' + name.upper().replace('.', '_') + '_PATH']))
    candidates += [
        Path(name), Path('outputs') / name, Path('data') / name,
        Path('../outputs') / name, Path('../data') / name,
        Path('/kaggle/working') / name,
    ]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(name))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(f'Không tìm thấy {name}.')

AGENT_EVIDENCE_PATH = find_data_file('agent_v4_results.json')
with AGENT_EVIDENCE_PATH.open(encoding='utf-8') as f:
    agent_evidence_raw = json.load(f)

MAX_EVIDENCE_CHARS = int(os.getenv('ALQAC_MAX_EVIDENCE_CHARS', '8000'))

def _case_evidence_text(record, max_chars=MAX_EVIDENCE_CHARS):
    seen, parts, total = set(), [], 0
    for e in record.get('evidence_details', []):
        chunk_id = e.get('chunk_id')
        txt = str(e.get('text', '')).strip()
        if not txt or chunk_id in seen:
            continue
        seen.add(chunk_id)
        if total + len(txt) + 1 > max_chars:
            break
        parts.append(txt)
        total += len(txt) + 1
    return '\n'.join(parts)

evidence_by_case = {r['case_id']: _case_evidence_text(r) for r in agent_evidence_raw}


def _case_evidence_ids(record):
    """ID cac segment evidence cua case, GIU NGUYEN thu tu, bo trung.

    'case_evidence' trong submission la danh sach id segment (vd 'case_4101_seg_...').
    Uu tien khoa 'evidence' cua agent_v4; neu thieu thi lay chunk_id trong 'evidence_details'.
    """
    ids, seen = [], set()
    for value in (record.get('evidence') or []):
        sid = str(value).strip()
        if sid and sid not in seen:
            seen.add(sid)
            ids.append(sid)
    if not ids:
        for detail in (record.get('evidence_details') or []):
            sid = str(detail.get('chunk_id', '')).strip()
            if sid and sid not in seen:
                seen.add(sid)
                ids.append(sid)
    return ids


evidence_ids_by_case = {r['case_id']: _case_evidence_ids(r) for r in agent_evidence_raw}
_missing_ev = [x['case_id'] for x in public_data if not evidence_by_case.get(x['case_id'])]
if _missing_ev:
    print('CẢNH BÁO: thiếu evidence cho case:', _missing_ev)
print('Evidence bổ sung:', AGENT_EVIDENCE_PATH, '| phủ', len(public_data) - len(_missing_ev), '/', len(public_data), 'case')

# Đây là view duy nhất được pipeline dự đoán sử dụng. Gold được giữ riêng cho cell đánh giá.
inference_cases = [
    {'case_id': x['case_id'], 'case_query': x['case_query'],
     'case_facts': evidence_by_case.get(x['case_id'], '')}
    for x in public_data
]
gold_by_case = {x['case_id']: x['verdict_label'] for x in public_data}

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)
if not qdrant.collection_exists(QDRANT_COLLECTION):
    raise RuntimeError(
        f'Chưa có collection {QDRANT_COLLECTION}. Hãy chạy notebook BGE-M3 trước.'
    )
collection_info = qdrant.get_collection(QDRANT_COLLECTION)
stored_dim = int(collection_info.config.params.vectors.size)
stored_count = int(qdrant.count(QDRANT_COLLECTION, exact=True).count)
assert stored_dim == EXPECTED_VECTOR_DIM, (stored_dim, EXPECTED_VECTOR_DIM)
assert stored_count == 3352, f'Expected 3352 laws, got {stored_count}'

embedder = SentenceTransformer(BGE_MODEL_ID, device=EMBED_DEVICE)
embedder.max_seq_length = 8192

print('Dataset:', DATA_PATH)
print('Cases:', len(inference_cases))
print('Qdrant:', QDRANT_COLLECTION, '| dim:', stored_dim, '| points:', stored_count)


Evidence bổ sung: /root/agent_v4_results.json | phủ 50 / 50 case


/usr/local/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.17.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Dataset: /root/ALQAC2026_public_test.json
Cases: 50
Qdrant: laws_bge_m3_v2_correct_pooling | dim: 1024 | points: 3352


In [4]:
def retrieve_laws(case_query, case_facts='', top_k=TOP_K):
    # (2026-08-11) Embed CA case_query + case_facts (chunks agent_v4) -- thu nghiem
    # "full chunk" thay vi chi query trơn, khop thuc nghiem pre_retrieval/eval-retrieval-bge.ipynb.
    embed_text = (case_query.strip() + '\n\n' + case_facts.strip()).strip() if case_facts else case_query.strip()
    query_vector = embedder.encode(
        [embed_text or ' '], normalize_embeddings=True, convert_to_numpy=True
    )[0]
    assert query_vector.shape == (EXPECTED_VECTOR_DIM,)
    assert np.isfinite(query_vector).all()
    hits = qdrant.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector.tolist(),
        limit=top_k,
        with_payload=True,
    ).points
    laws = []
    seen = set()
    for rank, hit in enumerate(hits, 1):
        payload = hit.payload or {}
        key = (str(payload.get('law_id')), int(payload.get('aid')))
        if key in seen:
            continue
        seen.add(key)
        laws.append({
            'rank': rank,
            'score': float(hit.score),
            'law_id': key[0],
            'aid': key[1],
            'article_no': int(payload.get('article_no')),
            'content_Article': str(payload.get('content_Article') or ''),
        })
    assert len(laws) == top_k, f'Chỉ retrieve được {len(laws)}/{top_k} luật'
    return laws

sample_case = inference_cases[0]
sample_laws = retrieve_laws(sample_case['case_query'], sample_case.get('case_facts', ''))
display(pd.DataFrame(sample_laws)[['rank', 'score', 'law_id', 'article_no', 'aid']])

# Retrieve đủ 50 case trước, rồi giải phóng BGE-M3 để dành toàn bộ VRAM cho LLM full precision.
retrieval_path = OUTPUT_DIR / 'retrieval_top10_bge_m3.json'
retrieval_meta_path = OUTPUT_DIR / 'retrieval_top10_bge_m3.meta.json'
retrieval_signature_material = {
    'bge_model': BGE_MODEL_ID, 'collection': QDRANT_COLLECTION, 'top_k': TOP_K,
    'cases': inference_cases,
}
retrieval_signature = hashlib.sha256(
    json.dumps(retrieval_signature_material, ensure_ascii=False, sort_keys=True).encode('utf-8')
).hexdigest()
cached_signature = None
if retrieval_meta_path.exists():
    try:
        cached_signature = json.loads(retrieval_meta_path.read_text(encoding='utf-8')).get('signature')
    except Exception:
        pass
if retrieval_path.exists() and cached_signature == retrieval_signature:
    try:
        retrieval_cache = json.loads(retrieval_path.read_text(encoding='utf-8'))
    except Exception:
        retrieval_cache = {}
else:
    retrieval_cache = {}
for case in tqdm(inference_cases, desc='BGE-M3 retrieval'):
    if case['case_id'] not in retrieval_cache:
        retrieval_cache[case['case_id']] = retrieve_laws(case['case_query'], case.get('case_facts', ''))
        temp_path = retrieval_path.with_suffix('.json.tmp')
        temp_path.write_text(json.dumps(retrieval_cache, ensure_ascii=False, indent=2), encoding='utf-8')
        temp_path.replace(retrieval_path)
retrieval_meta_path.write_text(
    json.dumps({'signature': retrieval_signature, **retrieval_signature_material}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
expected_case_ids = {x['case_id'] for x in inference_cases}
assert set(retrieval_cache) == expected_case_ids
assert all(len(retrieval_cache[cid]) == TOP_K for cid in expected_case_ids)

del embedder
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('Da cache retrieval va giai phong BGE-M3. GPU allocated:', round(torch.cuda.memory_allocated()/2**30, 2), 'GB')
else:
    print('Da cache retrieval va giai phong BGE-M3 (embedding chay tren CPU).')


,rank,score,law_id,article_no,aid
0,1,0.555824,91/2015/QH13,603,53373
1,2,0.532484,100/2015/QH13,266,56710
2,3,0.520990,91/2015/QH13,586,53356
3,4,0.516275,326/2016/UBTVQH14,27,13626
4,5,0.509900,100/2015/QH13,262,56706
5,6,0.508516,52/2014/QH13,74,53946
6,7,0.507149,100/2015/QH13,264,56708
7,8,0.507031,91/2015/QH13,595,53365
8,9,0.503605,93/2015/QH13,7,14312
9,10,0.502106,92/2015/QH13,359,51024


BGE-M3 retrieval:   0%|          | 0/50 [00:00<?, ?it/s]

Da cache retrieval va giai phong BGE-M3. GPU allocated: 0.01 GB


In [5]:
# ==== Robust compact structured vLLM engine — LOAD TRUC TIEP tren GPU cua kernel ====
# (2026-08-10) DOI KIEN TRUC: ban goc dung modal.App(...).run() de spin 1 container GPU
# RIENG (remote). Gap loi NotFoundError('Environment not found') tren tai khoan Modal
# cua nhom (xac nhan qua test toi gian, khong lien quan gi code pado) -- kha nang do
# cau hinh workspace/token, KHONG sua duoc tu code. Kernel Notebook nay DA CO SAN GPU
# dung duoc that -- bo han modal.App/RemoteLLM/app.run(), load thang vLLM engine trong
# CHINH tien trinh kernel. Giu NGUYEN 100% logic sampling/schema/usage cua ban v9 nay
# (vllm_generate KHONG co tham so enable_thinking rieng, luon dung ENABLE_THINKING
# toan cuc -- giu dung nhu vay, chi doi tu goi .remote() sang goi thang local).
import unicodedata

prompt_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, token=HF_TOKEN, revision=MODEL_REVISION, trust_remote_code=True
)

MODEL_DTYPE = VLLM_DTYPE
os.environ.setdefault('VLLM_USE_FLASHINFER_SAMPLER', '0')

GPU_TYPE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (KHONG CO GPU!)'
print(f'Loading {MODEL_ID} truc tiep tren GPU kernel ({GPU_TYPE}), khong qua Modal remote container...')
from vllm import LLM as _VllmLLM
vllm_engine = _VllmLLM(
    model=MODEL_ID,
    tokenizer=MODEL_ID,
    revision=MODEL_REVISION,
    tokenizer_revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
    trust_remote_code=True,
    gpu_memory_utilization=GPU_MEM_UTIL,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=MAX_NUM_SEQS,
    enforce_eager=VLLM_ENFORCE_EAGER,
    seed=EVAL_SEEDS[0],
)
print('vLLM ready (local, cung tien trinh voi notebook).')


def vllm_generate(messages_list, seeds, schema, max_tokens):
    if len(messages_list) != len(seeds):
        raise ValueError('messages_list and seeds length mismatch')
    from vllm import SamplingParams
    try:
        from vllm.sampling_params import StructuredOutputsParams
    except ImportError:
        from vllm import StructuredOutputsParams

    structured = None
    if USE_STRUCTURED_OUTPUTS and schema is not None:
        structured = StructuredOutputsParams(
            json=schema,
            disable_any_whitespace=True,
        )

    def make_params(seed):
        kwargs = dict(max_tokens=max_tokens, seed=int(seed))
        for _k in ('temperature', 'top_p', 'top_k', 'min_p', 'repetition_penalty'):
            if _k in SAMPLING:
                kwargs[_k] = SAMPLING[_k]
        if structured is not None:
            kwargs['structured_outputs'] = structured
        return SamplingParams(**kwargs)

    tmpl_kwargs = {}
    if 'qwen3' in MODEL_ID.lower():
        tmpl_kwargs['enable_thinking'] = ENABLE_THINKING
    outputs = vllm_engine.chat(
        messages_list,
        [make_params(seed) for seed in seeds],
        add_generation_prompt=True,
        chat_template_kwargs=tmpl_kwargs,
        use_tqdm=True,
    )
    results = []
    for output in outputs:
        generated = output.outputs[0]
        results.append((generated.text, {
            'input_tokens': len(output.prompt_token_ids),
            'output_tokens': len(generated.token_ids),
            'total_tokens': len(output.prompt_token_ids) + len(generated.token_ids),
            'hit_max_new_tokens': generated.finish_reason == 'length',
            'finish_reason': generated.finish_reason,
        }))
    return results


def close_modal():
    """Giai phong VRAM local (giu nguyen ten ham -- cell dong o cuoi notebook dang goi
    close_modal(), khong doi ten de khong phai sua cell khac)."""
    global vllm_engine
    try:
        del vllm_engine
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('Da giai phong GPU local.')


MODEL_CONTEXT_LIMIT = MAX_MODEL_LEN
RESOLVED_GENERATION_CONFIG = {
    'engine': 'vllm-local',
    'structured_outputs': USE_STRUCTURED_OUTPUTS,
    'disable_any_whitespace': USE_STRUCTURED_OUTPUTS,
    'enable_thinking': ENABLE_THINKING,
    'sampling': SAMPLING,
    'judge_votes': JUDGE_VOTES,
    'max_new_tokens_by_stage': {
        'issue_mapper': MAX_NEW_TOKENS_ISSUE,
        'legal_judge_vote': MAX_NEW_TOKENS_JUDGE,
        'direct_fallback': MAX_NEW_TOKENS_DIRECT,
    },
}

LABEL_GUIDE = '''Nhãn cuối:
- A_WIN: A được chấp nhận toàn bộ hoặc gần như toàn bộ phần MATERIAL.
- B_WIN: mọi hoặc gần như mọi phần MATERIAL của A bị bác.
- PARTIAL_A_WIN: cả hai bên giữ phần có ý nghĩa nhưng A nhận phần lớn giá trị MATERIAL.
- PARTIAL_B_WIN: A nhận một phần có ý nghĩa nhưng B giữ phần lớn giá trị MATERIAL.'''

GROUNDING_RULES = f'''Chỉ dùng CASE_QUERY và {TOP_K} điều luật truy xuất; A là nguyên đơn, B là bị đơn.
Xem các tình tiết được kể trong CASE_QUERY là dữ kiện, NHƯNG mức tiền/định lượng mà A yêu cầu chỉ là ĐÒI HỎI phải được xét, KHÔNG phải phần toà đã chấp nhận.
Thông tin không xuất hiện là UNKNOWN: không dùng sự im lặng để tự động ACCEPT hoặc REJECT, và không đòi thêm tài liệu chỉ vì bản tóm tắt không liệt kê.
Thực tế xét xử dân sự Việt Nam: toà tách TRÁCH NHIỆM (B có phải bồi thường/thực hiện nghĩa vụ không) khỏi ĐỊNH LƯỢNG (mức tiền cụ thể). Ngay cả khi trách nhiệm của B rõ, toà RẤT THƯỜNG chỉ chấp nhận MỘT PHẦN số tiền A đòi (giảm khoản thiếu chứng từ, lỗi hỗn hợp, khấu trừ nghĩa vụ đối ứng). Vì vậy một yêu cầu bồi thường/thiệt hại bằng tiền mà mức đòi chưa chắc chắn nên nghiêng PARTIAL, KHÔNG mặc định chấp nhận toàn bộ.'''

ISSUE_MAPPER_SYSTEM_PROMPT = f'''Bạn là Issue Mapper trung lập.
Chỉ inventory relief mà A thực sự yêu cầu trong CASE_QUERY. Không tạo legal issue, defense hay counterclaim thành claim.
Tách RIÊNG từng khoản có thể được toà quyết định độc lập: tiền gốc, lãi, phạt, TỪNG khoản thiệt hại/chi phí khác nhau, hủy/công nhận giao dịch, trả tài sản. Ví dụ "chi phí sửa xe" và "viện phí" là HAI claim tách biệt. Khi CASE_QUERY nối nhiều khoản bằng "và" hoặc dấu phẩy, hãy tách thành nhiều claim.
request_quote phải là đoạn liên tục copy từ CASE_QUERY (mỗi claim trích đúng phần của mình).
MATERIAL gồm tiền gốc, quyền/tài sản/giao dịch chính, trả tài sản, hủy hoặc công nhận giao dịch.
SECONDARY gồm lãi, phạt, án phí và chi phí phụ.
{GROUNDING_RULES}
Chỉ trả JSON theo schema.'''

JUDGE_SYSTEM_PROMPT = f'''Bạn là Legal Judge dự đoán kết quả tranh chấp dân sự Việt Nam.
Đọc kỹ CASE_QUERY và luật; issue map chỉ là inventory hỗ trợ, có thể chưa hoàn hảo.
BẮT BUỘC cân nhắc hai chiều trước khi quyết mỗi claim: (1) căn cứ MẠNH NHẤT để A được chấp nhận; (2) căn cứ MẠNH NHẤT để toà GIẢM hoặc BÁC một phần (thiếu chứng từ chứng minh mức tiền, lỗi hỗn hợp, mức đòi cao hơn thiệt hại thực, nghĩa vụ đối ứng của A). Chỉ FULL_ACCEPT khi CẢ trách nhiệm LẪN mức yêu cầu đều gần như chắc chắn.
Mỗi claim dùng một outcome (phải trung thực vì nhãn cuối được suy ra từ chúng):
- FULL_ACCEPT: A gần như chắc chắn được chấp nhận TOÀN BỘ claim (cả trách nhiệm và mức tiền).
- PARTIAL_A_LEAN: A thắng phần lớn nhưng khả năng bị giảm/không toàn bộ (điển hình cho yêu cầu bồi thường bằng tiền còn tranh cãi về mức).
- PARTIAL_B_LEAN: claim bị chia nhưng B giữ phần lớn hơn.
- REJECT: A gần như bị bác toàn bộ claim.
Không kết luận FULL_ACCEPT chỉ vì thấy trách nhiệm của B; hãy hỏi mức tiền có chắc chắn không. Không kết luận REJECT chỉ vì bản tóm tắt thiếu tài liệu.
Ưu tiên điều luật khớp trực tiếp tình tiết; đánh giá theo GIÁ TRỊ phần MATERIAL, không đếm claim thô.
{LABEL_GUIDE}
{GROUNDING_RULES}
Trả JSON theo schema: claim_outcomes (mỗi claim một outcome + legal_basis_ranks), prediction (nhãn tổng thể nhất quán với các outcome), confidence.'''

DIRECT_SYSTEM_PROMPT = f'''Bạn là bộ phân loại dự phòng cho tranh chấp dân sự Việt Nam.
Đọc CASE_QUERY và {TOP_K} điều luật rồi chọn đúng một nhãn. Không dùng gold label hoặc dữ liệu đánh giá.
{LABEL_GUIDE}
{GROUNDING_RULES}
Chỉ trả compact JSON theo schema.'''


def build_law_context(laws):
    blocks = []
    for law in laws:
        content = law['content_Article'][:MAX_ARTICLE_CHARS]
        blocks.append(
            f"[{law['rank']}] law_id={law['law_id']} | Điều {law['article_no']} | aid={law['aid']}\n"
            f'{content}'
        )
    return '\n\n'.join(blocks)


def build_issue_prompt(case_query, laws, case_facts=''):
    facts_block = (
        '\n\nTHÔNG TIN VỤ VIỆC (trích đoạn hồ sơ, evidence bổ sung):\n' + case_facts.strip()
    ) if case_facts else ''
    return f'''CASE_QUERY:
{case_query.strip()}{facts_block}

{TOP_K} ĐIỀU LUẬT:
{build_law_context(laws)}

Inventory các relief của A theo thứ tự xuất hiện.'''


def build_judge_prompt(case_query, laws, issue_map, case_facts=''):
    facts_block = (
        '\n\nTHÔNG TIN VỤ VIỆC (trích đoạn hồ sơ, evidence bổ sung):\n' + case_facts.strip()
    ) if case_facts else ''
    return f'''CASE_QUERY:
{case_query.strip()}{facts_block}

{TOP_K} ĐIỀU LUẬT:
{build_law_context(laws)}

ISSUE MAP:
{json.dumps(issue_map, ensure_ascii=False, separators=(',', ':'))}

Quyết định từng claim và prediction tổng thể. prediction phải phản ánh giá trị phần MATERIAL, không phải số claim.'''


def build_direct_prompt(case_query, laws, case_facts=''):
    facts_block = (
        '\n\nTHÔNG TIN VỤ VIỆC (trích đoạn hồ sơ, evidence bổ sung):\n' + case_facts.strip()
    ) if case_facts else ''
    return f'''CASE_QUERY:
{case_query.strip()}{facts_block}

{TOP_K} ĐIỀU LUẬT:
{build_law_context(laws)}

Chọn nhãn dự đoán cuối cùng.'''


def build_messages(system_prompt, user_prompt):
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]


def count_message_tokens(messages):
    return len(prompt_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
    ))


def _normalize_match(text):
    normalized = unicodedata.normalize('NFKC', str(text or '')).casefold()
    return re.sub(r'[\W_]+', ' ', normalized, flags=re.UNICODE).strip()


def _is_query_span(candidate, case_query):
    candidate_norm = _normalize_match(candidate)
    return bool(candidate_norm) and candidate_norm in _normalize_match(case_query)


def fallback_issue_map(case_query, reason):
    return {
        'claims': [{
            'claim_id': 'C1',
            'request_quote': case_query.strip(),
            'request_type': 'OTHER',
            'importance': 'MATERIAL',
        }],
        'mapper_fallback': True,
        'mapper_fallback_reason': str(reason),
    }


def validate_issue_map(obj, case_query):
    raw_claims = obj.get('claims')
    if not isinstance(raw_claims, list):
        raise ValueError('claims must be a list')
    valid_types = {'PRINCIPAL', 'INTEREST', 'PENALTY', 'DAMAGES', 'PROPERTY', 'TRANSACTION', 'OTHER'}
    claims, seen = [], set()
    for raw in raw_claims:
        if not isinstance(raw, dict):
            continue
        quote = str(raw.get('request_quote', '')).strip()
        if not _is_query_span(quote, case_query):
            continue
        request_type = str(raw.get('request_type', '')).strip().upper()
        importance = str(raw.get('importance', '')).strip().upper()
        if request_type not in valid_types or importance not in {'MATERIAL', 'SECONDARY'}:
            continue
        key = (_normalize_match(quote), request_type)
        if key in seen:
            continue
        seen.add(key)
        claims.append({
            'claim_id': f'C{len(claims) + 1}',
            'request_quote': quote,
            'request_type': request_type,
            'importance': importance,
        })
    if not claims:
        raise ValueError('no grounded unique claims survived validation')
    if not any(claim['importance'] == 'MATERIAL' for claim in claims):
        claims[0]['importance'] = 'MATERIAL'
    return {'claims': claims, 'mapper_fallback': False, 'mapper_fallback_reason': None}


def _json_object(text):
    cleaned = (text or '')
    # Thinking mode phát ra <think>...</think> trước JSON; bỏ đi như base.
    cleaned = re.sub(r'<think>.*?</think>', '', cleaned, flags=re.I | re.S)
    if '</think>' in cleaned:
        cleaned = cleaned.rsplit('</think>', 1)[-1]
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', cleaned.strip(), flags=re.I | re.S).strip()
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    decoder = json.JSONDecoder()
    for match in re.finditer(r'\{', cleaned):
        try:
            obj, _ = decoder.raw_decode(cleaned[match.start():])
        except Exception:
            continue
        if isinstance(obj, dict):
            return obj
    raise ValueError('no complete JSON object')


def derive_label(issue_map, claim_outcomes):
    '''Suy nhãn 4 lớp từ outcome của từng claim, trọng số theo importance.
    PADO đúng nghĩa: nhãn tổng thể là hệ quả của các quyết định claim, không phải
    nhãn model tự khai. Đây là điều làm PARTIAL xuất hiện đúng cấu trúc.'''
    weight_by_id = {
        c['claim_id']: (2.0 if c.get('importance') == 'MATERIAL' else 1.0)
        for c in issue_map['claims']
    }
    a_share = {'FULL_ACCEPT': 1.0, 'PARTIAL_A_LEAN': 0.65, 'PARTIAL_B_LEAN': 0.35, 'REJECT': 0.0}
    num = den = 0.0
    for co in claim_outcomes:
        w = weight_by_id.get(co['claim_id'], 1.0)
        num += w * a_share.get(co['outcome'], 0.5)
        den += w
    if den == 0:
        return None
    s = num / den
    if s >= 0.85:
        return 'A_WIN'
    if s >= 0.55:
        return 'PARTIAL_A_WIN'
    if s >= 0.30:
        return 'PARTIAL_B_WIN'
    return 'B_WIN'


def validate_judge(obj, issue_map, retrieved_laws):
    prediction = str(obj.get('prediction', '')).strip().upper()
    if prediction not in LABELS:
        raise ValueError(f'invalid prediction {prediction!r}')
    try:
        confidence = min(1.0, max(0.0, float(obj.get('confidence', 0.0))))
    except Exception:
        confidence = 0.0

    valid_outcomes = {'FULL_ACCEPT', 'PARTIAL_A_LEAN', 'PARTIAL_B_LEAN', 'REJECT'}
    expected_ids = [claim['claim_id'] for claim in issue_map['claims']]
    raw_by_id = {}
    for item in obj.get('claim_outcomes', []) if isinstance(obj.get('claim_outcomes'), list) else []:
        if not isinstance(item, dict):
            continue
        claim_id = str(item.get('claim_id', '')).strip()
        outcome = str(item.get('outcome', '')).strip().upper()
        if claim_id not in expected_ids or outcome not in valid_outcomes or claim_id in raw_by_id:
            continue
        ranks = []
        for rank in item.get('legal_basis_ranks', []) if isinstance(item.get('legal_basis_ranks'), list) else []:
            try:
                rank = int(rank)
            except Exception:
                continue
            if 1 <= rank <= len(retrieved_laws) and rank not in ranks:
                ranks.append(rank)
        raw_by_id[claim_id] = {
            'claim_id': claim_id,
            'outcome': outcome,
            'legal_basis_ranks': ranks[:3],
        }

    default_outcome = {
        'A_WIN': 'FULL_ACCEPT',
        'PARTIAL_A_WIN': 'PARTIAL_A_LEAN',
        'PARTIAL_B_WIN': 'PARTIAL_B_LEAN',
        'B_WIN': 'REJECT',
    }[prediction]
    outcomes = [
        raw_by_id.get(claim_id, {
            'claim_id': claim_id,
            'outcome': default_outcome,
            'legal_basis_ranks': [],
        })
        for claim_id in expected_ids
    ]
    ranks = []
    for item in outcomes:
        for rank in item['legal_basis_ranks']:
            if rank not in ranks:
                ranks.append(rank)
    applied_laws = [
        {
            'law_id': str(retrieved_laws[rank - 1]['law_id']),
            'aid': int(retrieved_laws[rank - 1]['aid']),
            'reason': f'Legal Judge selected retrieved law rank {rank}',
        }
        for rank in ranks
    ]
    derived = derive_label(issue_map, outcomes)
    # Nhãn quyết định = phán quyết tổng thể của Judge (ensemble majority ở aggregate_votes).
    # derive_label chỉ lưu để audit: vì mapper thực tế luôn tạo 1 claim, việc suy nhãn từ
    # outcome đơn làm sập dự đoán về PARTIAL_A_WIN (public 50: strict 0.36, macroF 0.174),
    # kém holistic (strict 0.40, macroF 0.251). Nên KHÔNG dùng derived làm decider.
    return {
        'prediction': prediction,
        'derived_label': derived,
        'confidence': confidence,
        'claim_outcomes': outcomes,
        'applied_laws': applied_laws,
    }


def validate_direct(obj):
    prediction = str(obj.get('prediction', '')).strip().upper()
    if prediction not in LABELS:
        raise ValueError(f'invalid direct prediction {prediction!r}')
    try:
        confidence = min(1.0, max(0.0, float(obj.get('confidence', 0.0))))
    except Exception:
        confidence = 0.0
    return {'prediction': prediction, 'confidence': confidence}


def aggregate_votes(votes):
    if not votes:
        raise ValueError('cannot aggregate zero votes')
    counts = {label: 0 for label in LABELS}
    confidence_sum = {label: 0.0 for label in LABELS}
    for vote in votes:
        label = vote['prediction']
        counts[label] += 1
        confidence_sum[label] += vote['confidence']
    max_count = max(counts.values())
    candidates = [label for label in LABELS if counts[label] == max_count]
    candidates.sort(
        key=lambda label: (
            confidence_sum[label] / max(1, counts[label]),
            -LABELS.index(label),
        ),
        reverse=True,
    )
    prediction = candidates[0]
    selected = [vote for vote in votes if vote['prediction'] == prediction]
    representative = max(selected, key=lambda vote: vote['confidence'])
    return {
        'prediction': prediction,
        'confidence': sum(vote['confidence'] for vote in selected) / len(selected),
        'vote_counts': counts,
        'n_valid_votes': len(votes),
        'claim_outcomes': representative['claim_outcomes'],
        'applied_laws': representative['applied_laws'],
    }


def deterministic_stage_seed(eval_seed, case_id, stage):
    raw = f'{eval_seed}|{case_id}|{stage}'.encode('utf-8')
    return int.from_bytes(hashlib.sha256(raw).digest()[:4], 'big')


ISSUE_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'claims': {
            'type': 'array',
            'minItems': 1,
            'maxItems': 8,
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'properties': {
                    'claim_id': {'type': 'string', 'pattern': '^C[1-8]$'},
                    'request_quote': {'type': 'string', 'minLength': 1, 'maxLength': 600},
                    'request_type': {
                        'type': 'string',
                        'enum': ['PRINCIPAL', 'INTEREST', 'PENALTY', 'DAMAGES', 'PROPERTY', 'TRANSACTION', 'OTHER'],
                    },
                    'importance': {'type': 'string', 'enum': ['MATERIAL', 'SECONDARY']},
                },
                'required': ['claim_id', 'request_quote', 'request_type', 'importance'],
            },
        },
    },
    'required': ['claims'],
}

CLAIM_OUTCOME_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'claim_id': {'type': 'string', 'pattern': '^C[1-8]$'},
        'outcome': {
            'type': 'string',
            'enum': ['FULL_ACCEPT', 'PARTIAL_A_LEAN', 'PARTIAL_B_LEAN', 'REJECT'],
        },
        'legal_basis_ranks': {
            'type': 'array',
            'minItems': 1,
            'maxItems': 3,
            'items': {'type': 'integer', 'minimum': 1, 'maximum': 10},
        },
    },
    'required': ['claim_id', 'outcome', 'legal_basis_ranks'],
}

JUDGE_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'claim_outcomes': {'type': 'array', 'minItems': 1, 'maxItems': 8, 'items': CLAIM_OUTCOME_SCHEMA},
        'prediction': {'type': 'string', 'enum': LABELS},
        'confidence': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0},
    },
    'required': ['claim_outcomes', 'prediction', 'confidence'],
}

DIRECT_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'prediction': {'type': 'string', 'enum': LABELS},
        'confidence': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0},
    },
    'required': ['prediction', 'confidence'],
}

HARD_FALLBACK_LABEL = os.getenv('ALQAC_HARD_FALLBACK_LABEL', 'PARTIAL_A_WIN').strip().upper()
if HARD_FALLBACK_LABEL not in LABELS:
    raise ValueError(f'Invalid ALQAC_HARD_FALLBACK_LABEL={HARD_FALLBACK_LABEL!r}')

PIPELINE_CONTRACT = {
    'pipeline_version': PADO_PIPELINE_VERSION,
    'stages': ['issue_mapper', 'legal_judge_3vote', 'direct_fallback'],
    'issue_schema': ISSUE_SCHEMA,
    'judge_schema': JUDGE_SCHEMA,
    'direct_schema': DIRECT_SCHEMA,
    'judge_votes': JUDGE_VOTES,
    'disable_any_whitespace': USE_STRUCTURED_OUTPUTS,
    'hard_fallback_label': HARD_FALLBACK_LABEL,
}
PIPELINE_CONTRACT_HASH = hashlib.sha256(
    json.dumps(PIPELINE_CONTRACT, ensure_ascii=False, sort_keys=True).encode('utf-8')
).hexdigest()


def aggregate_stage_usage(stage_usages):
    values = list(stage_usages.values())
    return {
        'input_tokens': sum(item['input_tokens'] for item in values),
        'output_tokens': sum(item['output_tokens'] for item in values),
        'total_tokens': sum(item['total_tokens'] for item in values),
        'hit_max_new_tokens': any(item.get('hit_max_new_tokens', False) for item in values),
        'stage_count': len(values),
        'stages': stage_usages,
    }





BENCHMARK_MANIFEST = {
    'benchmark_version': BENCHMARK_VERSION,
    'benchmark_protocol': BENCHMARK_PROTOCOL,
    'pipeline_version': PADO_PIPELINE_VERSION,
    'pipeline_contract_hash': PIPELINE_CONTRACT_HASH,
    'pipeline_stages': PIPELINE_CONTRACT['stages'],
    'model_name': MODEL_NAME,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'generation': RESOLVED_GENERATION_CONFIG,
    'model_context_limit': MODEL_CONTEXT_LIMIT,
    'dtype': MODEL_DTYPE,
    'full_gpu_no_quantization': True,
    'dataset_path': str(DATA_PATH),
    'dataset_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'num_cases': len(inference_cases),
    'labels': LABELS,
    'bge_model': BGE_MODEL_ID,
    'qdrant_collection': QDRANT_COLLECTION,
    'top_k_laws': TOP_K,
    'retrieval_signature': retrieval_signature,
    'max_input_tokens': MAX_INPUT_TOKENS,
    'max_article_chars': MAX_ARTICLE_CHARS,
    'eval_seeds': EVAL_SEEDS,
    'coverage_policy': 'mapper_fallback + direct_fallback + explicit hard fallback always emit a label',
    'prompt_hashes': {
        'issue_mapper': hashlib.sha256(ISSUE_MAPPER_SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
        'legal_judge': hashlib.sha256(JUDGE_SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
        'direct_fallback': hashlib.sha256(DIRECT_SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
    },
}

print(
    'v9 compact ensemble ready | GPU:', GPU_TYPE,
    '| whitespace disabled: True',
    '| thinking:', ENABLE_THINKING,
    '| sampling:', SAMPLING,
    '| votes:', JUDGE_VOTES,
    '| contract:', PIPELINE_CONTRACT_HASH[:12],
)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

WARNING 08-11 15:01:48 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=486) INFO 08-11 15:02:08 [core.py:114] Initializing a V1 LLM engine (v0.25.1) with config: model='Qwen/Qwen3-4B', speculative_config=None, tokenizer='Qwen/Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=1cfa9a7208912126459214e8b04321603b3df60c, tokenizer_revision=1cfa9a7208912126459214e8b04321603b3df60c, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cac

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.22it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.22it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  1.81it/s]
(EngineCore pid=486) 


(EngineCore pid=486) INFO 08-11 15:03:06 [default_loader.py:430] Loading weights took 1.69 seconds
(EngineCore pid=486) INFO 08-11 15:03:07 [model_runner.py:302] Model loading took 7.56 GiB and 55.280523 seconds
(EngineCore pid=486) INFO 08-11 15:03:07 [topk_topp_sampler.py:39] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=486) INFO 08-11 15:03:25 [backends.py:1089] Using cache directory: /root/.cache/vllm/torch_compile_cache/2a9a326e2e/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=486) INFO 08-11 15:03:25 [backends.py:1148] Dynamo bytecode transform time: 18.28 s
(EngineCore pid=486) INFO 08-11 15:03:38 [backends.py:378] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=486) INFO 08-11 15:03:50 [backends.py:393] Compiling a graph for compile range (1, 8192) takes 24.48 s
(EngineCore pid=486) INFO 08-11 15:04:03 [decorators.py:708] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch

Capturing CUDA graphs (FULL): 100%|████████████████████████████████████████████| 11/11 [00:01<00:00,  9.11it/s]


(EngineCore pid=486) INFO 08-11 15:04:11 [model_runner.py:722] Graph capturing finished in 5 secs, took 0.23 GiB
(EngineCore pid=486) INFO 08-11 15:04:32 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=486) INFO 08-11 15:04:33 [core.py:337] init engine (profile, create kv cache, warmup model) took 86.29 s (compilation: 55.83 s)
(EngineCore pid=486) INFO 08-11 15:04:34 [vllm.py:1042] Asynchronous scheduling is enabled.
(EngineCore pid=486) INFO 08-11 15:04:34 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
vLLM ready (local, cung tien trinh voi notebook).
v9 compact ensemble ready | GPU: NVIDIA A100-SXM4-40GB | whitespace disabled: True | thinking: True | sampling: {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20} | votes: 3 | contract: 514a2a1431f1


In [6]:
_cid = sample_case['case_id']
_query = sample_case['case_query']
_facts = sample_case.get('case_facts', '')
_seed = EVAL_SEEDS[0]
_laws = retrieval_cache[_cid]
try:
    mapper_raw, mapper_usage = vllm_generate(
        [build_messages(ISSUE_MAPPER_SYSTEM_PROMPT, build_issue_prompt(_query, _laws, _facts))],
        [deterministic_stage_seed(_seed, _cid, 'issue_mapper')],
        ISSUE_SCHEMA,
        MAX_NEW_TOKENS_ISSUE,
    )[0]
    try:
        issue_map = validate_issue_map(_json_object(mapper_raw), _query)
    except Exception as mapper_exc:
        issue_map = fallback_issue_map(_query, mapper_exc)

    vote_raw, vote_usage = vllm_generate(
        [build_messages(JUDGE_SYSTEM_PROMPT, build_judge_prompt(_query, _laws, issue_map, _facts))],
        [deterministic_stage_seed(_seed, _cid, 'legal_judge_vote_0')],
        JUDGE_SCHEMA,
        MAX_NEW_TOKENS_JUDGE,
    )[0]
    vote = validate_judge(_json_object(vote_raw), issue_map, _laws)
    print(
        'SMOKE OK | mapper_fallback:', issue_map['mapper_fallback'],
        '| claims:', len(issue_map['claims']),
        '| prediction:', vote['prediction'],
        '| usage:', aggregate_stage_usage({'issue_mapper': mapper_usage, 'legal_judge_vote_0': vote_usage}),
    )
except Exception as exc:
    print('SMOKE WARNING:', repr(exc))


Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 08-11 15:04:47 [hf.py:548] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██| 1/1 [00:20<00:00, 20.86s/it, est. speed input: 554.10 toks/s, output: 97.37 toks/s]


Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|█| 1/1 [00:10<00:00, 10.56s/it, est. speed input: 1132.44 toks/s, output: 95.36 toks/s]

SMOKE OK | mapper_fallback: True | claims: 1 | prediction: PARTIAL_A_WIN | usage: {'input_tokens': 23517, 'output_tokens': 3038, 'total_tokens': 26555, 'hit_max_new_tokens': False, 'stage_count': 2, 'stages': {'issue_mapper': {'input_tokens': 11558, 'output_tokens': 2031, 'total_tokens': 13589, 'hit_max_new_tokens': False, 'finish_reason': 'stop'}, 'legal_judge_vote_0': {'input_tokens': 11959, 'output_tokens': 1007, 'total_tokens': 12966, 'hit_max_new_tokens': False, 'finish_reason': 'stop'}}}


In [7]:
def slugify(text):
    return re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')


def load_json(path, default):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return default


def atomic_write_json(path, obj):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)


def make_cache_key(model, case, laws, eval_seed):
    material = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'pipeline_version': PADO_PIPELINE_VERSION,
        'pipeline_contract_hash': PIPELINE_CONTRACT_HASH,
        'model': model,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'generation': RESOLVED_GENERATION_CONFIG,
        'eval_seed': eval_seed,
        'case_id': case['case_id'],
        'case_query': case['case_query'],
        'case_facts': case.get('case_facts', ''),
        'laws': laws,
        'prompts': {
            'issue_mapper': ISSUE_MAPPER_SYSTEM_PROMPT,
            'legal_judge': JUDGE_SYSTEM_PROMPT,
            'direct_fallback': DIRECT_SYSTEM_PROMPT,
        },
        'schemas': {'issue': ISSUE_SCHEMA, 'judge': JUDGE_SCHEMA, 'direct': DIRECT_SCHEMA},
    }
    return hashlib.sha256(
        json.dumps(material, ensure_ascii=False, sort_keys=True).encode('utf-8')
    ).hexdigest()


manifest_path = OUTPUT_DIR / f'benchmark_manifest_{slugify(MODEL_NAME)}.json'
atomic_write_json(manifest_path, BENCHMARK_MANIFEST)
print('Benchmark manifest:', manifest_path)

MODEL = MODELS_TO_RUN[0]
EVAL_SEED = EVAL_SEEDS[0]
result_path = OUTPUT_DIR / f'predictions_{slugify(MODEL)}_seed-{EVAL_SEED}.json'
saved = load_json(result_path, {})

cache_keys, cases_run = {}, []
for case in inference_cases:
    cid = case['case_id']
    cache_key = make_cache_key(MODEL, case, retrieval_cache[cid], EVAL_SEED)
    cache_keys[cid] = cache_key
    old = saved.get(cid)
    if old and old.get('cache_key') == cache_key and old.get('prediction') in LABELS:
        continue
    cases_run.append(case)
print(f'=== v9 compact ensemble | seed={EVAL_SEED} | run {len(cases_run)}/{len(inference_cases)} cases ===')

state = {}
for case in cases_run:
    state[case['case_id']] = {
        'laws': retrieval_cache[case['case_id']],
        'raw': {},
        'usage': {},
        'issue_map': None,
        'votes': [],
        'stage_errors': [],
    }


def checked_messages(system_prompt, user_prompt, max_tokens):
    messages = build_messages(system_prompt, user_prompt)
    input_tokens = count_message_tokens(messages)
    if input_tokens > MAX_INPUT_TOKENS or input_tokens + max_tokens > MODEL_CONTEXT_LIMIT:
        raise ValueError(
            f'prompt budget exceeded: input={input_tokens}, output={max_tokens}, '
            f'limits={MAX_INPUT_TOKENS}/{MODEL_CONTEXT_LIMIT}'
        )
    return messages


started_all = time.time()

# Stage 1: mapper. Every failure is converted to a one-claim grounded fallback.
mapper_messages, mapper_seeds, mapper_cases = [], [], []
for case in cases_run:
    current = state[case['case_id']]
    try:
        mapper_messages.append(checked_messages(
            ISSUE_MAPPER_SYSTEM_PROMPT,
            build_issue_prompt(case['case_query'], current['laws'], case.get('case_facts', '')),
            MAX_NEW_TOKENS_ISSUE,
        ))
        mapper_seeds.append(deterministic_stage_seed(EVAL_SEED, case['case_id'], 'issue_mapper'))
        mapper_cases.append(case)
    except Exception as exc:
        current['stage_errors'].append({'stage': 'issue_mapper_input', 'error': repr(exc)})
        current['issue_map'] = fallback_issue_map(case['case_query'], exc)

if mapper_cases:
    try:
        mapper_outputs = vllm_generate(
            mapper_messages, mapper_seeds, ISSUE_SCHEMA, MAX_NEW_TOKENS_ISSUE
        )
        if len(mapper_outputs) != len(mapper_cases):
            raise RuntimeError(f'Mapper returned {len(mapper_outputs)}/{len(mapper_cases)} outputs')
        for case, (text, usage) in zip(mapper_cases, mapper_outputs):
            current = state[case['case_id']]
            current['raw']['issue_mapper'] = text
            current['usage']['issue_mapper'] = usage
            try:
                current['issue_map'] = validate_issue_map(_json_object(text), case['case_query'])
            except Exception as exc:
                current['stage_errors'].append({'stage': 'issue_mapper_output', 'error': repr(exc)})
                current['issue_map'] = fallback_issue_map(case['case_query'], exc)
    except Exception as exc:
        for case in mapper_cases:
            current = state[case['case_id']]
            current['stage_errors'].append({'stage': 'issue_mapper_infrastructure', 'error': repr(exc)})
            current['issue_map'] = fallback_issue_map(case['case_query'], exc)

for case in cases_run:
    current = state[case['case_id']]
    if current['issue_map'] is None:
        current['issue_map'] = fallback_issue_map(case['case_query'], 'mapper produced no state')

# Stage 2: three independent compact Judge votes. A failed vote does not kill the case.
for vote_index in range(JUDGE_VOTES):
    vote_messages, vote_seeds, vote_cases = [], [], []
    for case in cases_run:
        current = state[case['case_id']]
        try:
            vote_messages.append(checked_messages(
                JUDGE_SYSTEM_PROMPT,
                build_judge_prompt(
                    case['case_query'], current['laws'], current['issue_map'], case.get('case_facts', '')
                ),
                MAX_NEW_TOKENS_JUDGE,
            ))
            vote_seeds.append(deterministic_stage_seed(
                EVAL_SEED, case['case_id'], f'legal_judge_vote_{vote_index}'
            ))
            vote_cases.append(case)
        except Exception as exc:
            current['stage_errors'].append({
                'stage': f'legal_judge_vote_{vote_index}_input', 'error': repr(exc)
            })
    if not vote_cases:
        continue
    try:
        vote_outputs = vllm_generate(
            vote_messages, vote_seeds, JUDGE_SCHEMA, MAX_NEW_TOKENS_JUDGE
        )
        if len(vote_outputs) != len(vote_cases):
            raise RuntimeError(f'Judge vote {vote_index} returned {len(vote_outputs)}/{len(vote_cases)} outputs')
        for case, (text, usage) in zip(vote_cases, vote_outputs):
            current = state[case['case_id']]
            stage = f'legal_judge_vote_{vote_index}'
            current['raw'][stage] = text
            current['usage'][stage] = usage
            try:
                vote = validate_judge(
                    _json_object(text), current['issue_map'], current['laws']
                )
                vote['vote_index'] = vote_index
                current['votes'].append(vote)
            except Exception as exc:
                current['stage_errors'].append({'stage': stage, 'error': repr(exc)})
    except Exception as exc:
        for case in vote_cases:
            state[case['case_id']]['stage_errors'].append({
                'stage': f'legal_judge_vote_{vote_index}_infrastructure', 'error': repr(exc)
            })

# Direct fallback only for cases with zero usable Judge votes.
direct_cases = [case for case in cases_run if not state[case['case_id']]['votes']]
if direct_cases:
    direct_messages, direct_seeds, runnable_direct = [], [], []
    for case in direct_cases:
        current = state[case['case_id']]
        try:
            direct_messages.append(checked_messages(
                DIRECT_SYSTEM_PROMPT,
                build_direct_prompt(case['case_query'], current['laws'], case.get('case_facts', '')),
                MAX_NEW_TOKENS_DIRECT,
            ))
            direct_seeds.append(deterministic_stage_seed(EVAL_SEED, case['case_id'], 'direct_fallback'))
            runnable_direct.append(case)
        except Exception as exc:
            current['stage_errors'].append({'stage': 'direct_fallback_input', 'error': repr(exc)})
    if runnable_direct:
        try:
            direct_outputs = vllm_generate(
                direct_messages, direct_seeds, DIRECT_SCHEMA, MAX_NEW_TOKENS_DIRECT
            )
            if len(direct_outputs) != len(runnable_direct):
                raise RuntimeError(f'Direct fallback returned {len(direct_outputs)}/{len(runnable_direct)} outputs')
            for case, (text, usage) in zip(runnable_direct, direct_outputs):
                current = state[case['case_id']]
                current['raw']['direct_fallback'] = text
                current['usage']['direct_fallback'] = usage
                try:
                    current['direct'] = validate_direct(_json_object(text))
                except Exception as exc:
                    current['stage_errors'].append({'stage': 'direct_fallback', 'error': repr(exc)})
        except Exception as exc:
            for case in runnable_direct:
                state[case['case_id']]['stage_errors'].append({
                    'stage': 'direct_fallback_infrastructure', 'error': repr(exc)
                })

duration_all = round(time.time() - started_all, 3)
per_case_duration = round(duration_all / max(1, len(cases_run)), 3)
for case in cases_run:
    cid = case['case_id']
    current = state[cid]
    if current['votes']:
        verdict = aggregate_votes(current['votes'])
        prediction_source = (
            'judge_ensemble' if len(current['votes']) >= 2 else 'judge_single_vote'
        )
    elif current.get('direct'):
        verdict = {
            **current['direct'],
            'vote_counts': {label: 0 for label in LABELS},
            'n_valid_votes': 0,
            'claim_outcomes': [],
            'applied_laws': [],
        }
        prediction_source = 'direct_fallback'
    else:
        verdict = {
            'prediction': HARD_FALLBACK_LABEL,
            'confidence': 0.0,
            'vote_counts': {label: 0 for label in LABELS},
            'n_valid_votes': 0,
            'claim_outcomes': [],
            'applied_laws': [],
        }
        prediction_source = 'hard_fallback'

    result = {
        'case_id': cid,
        'case_query': case['case_query'],
        'eval_seed': EVAL_SEED,
        'pipeline_version': PADO_PIPELINE_VERSION,
        'pipeline_contract_hash': PIPELINE_CONTRACT_HASH,
        'pipeline_stages': {
            'issue_mapper': current['issue_map'],
            'judge_votes': current['votes'],
        },
        'retrieved_laws': current['laws'],
        'raw_responses': current['raw'],
        'usage': aggregate_stage_usage(current['usage']),
        'duration_seconds': per_case_duration,
        'cache_key': cache_keys[cid],
        'prediction': verdict['prediction'],
        'prediction_source': prediction_source,
        'confidence': verdict['confidence'],
        'reasoning': (
            f"source={prediction_source}; votes={verdict['vote_counts']}; "
            f"valid_votes={verdict['n_valid_votes']}"
        ),
        'applied_laws': verdict['applied_laws'],
        'claim_outcomes': verdict['claim_outcomes'],
        'vote_counts': verdict['vote_counts'],
        'n_valid_votes': verdict['n_valid_votes'],
        'mapper_fallback': bool(current['issue_map'].get('mapper_fallback')),
        'stage_errors': current['stage_errors'],
        'failed_stage': None,
        'error_kind': None,
        'error': None,
    }
    saved[cid] = result

    # Checkpoint từng case: nếu notebook bị ngắt sau đó, rerun không phải sinh lại case đã hoàn tất.
    atomic_write_json(result_path, saved)

expected_ids = {case['case_id'] for case in inference_cases}
missing_ids = sorted(expected_ids - set(saved))
invalid_ids = sorted(
    cid for cid in expected_ids if (saved.get(cid) or {}).get('prediction') not in LABELS
)
if missing_ids or invalid_ids:
    raise RuntimeError(f'Coverage invariant failed: missing={missing_ids}, invalid={invalid_ids}')
atomic_write_json(result_path, saved)
all_model_results = {MODEL: {EVAL_SEED: saved}}
source_counts = {}
for value in saved.values():
    source = value.get('prediction_source', 'unknown')
    source_counts[source] = source_counts.get(source, 0) + 1
print(
    f'Completed v9: {sum(v.get("prediction") in LABELS for v in saved.values())}/'
    f'{len(inference_cases)} valid | sources={source_counts} | '
    f'{duration_all}s for {len(cases_run)} cases (~{per_case_duration}s/case).'
)


Benchmark manifest: outputs_alqac_e2e/v10_legal_pado_thinking_base_matched/benchmark_manifest_qwen3-4b.json
=== v9 compact ensemble | seed=2026 | run 50/50 cases ===


Rendering conversations:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|            | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=486) WARNING 08-11 15:06:19 [jit_monitor.py:129] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|█| 50/50 [03:07<00:00,  3.75s/it, est. speed input: 2703.40 toks/s, output: 414.84 toks


...

Rendering conversations:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts: 100%|█| 50/50 [02:20<00:00,  2.81s/it, est. speed input: 3758.84 toks/s, output: 432.49 toks


Rendering conversations:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts: 100%|█| 50/50 [04:49<00:00,  5.79s/it, est. speed input: 1826.59 toks/s, output: 269.03 toks


Completed v9: 50/50 valid | sources={'judge_ensemble': 50} | 768.758s for 50 cases (~15.375s/case).


In [8]:
def evaluate_run(model, eval_seed, result_map):
    rows = []
    for case in inference_cases:
        cid = case['case_id']
        item = result_map.get(cid, {})
        prediction = item.get('prediction')
        is_valid_output = prediction in LABELS
        usage = item.get('usage') or {}
        rows.append({
            'model': model,
            'seed': eval_seed,
            'case_id': cid,
            'gold': gold_by_case[cid],
            'prediction': prediction,
            'scored_prediction': prediction if is_valid_output else INVALID_OUTPUT_LABEL,
            'is_valid_output': is_valid_output,
            'error_kind': item.get('error_kind'),
            'is_infrastructure_failure': item.get('error_kind') == 'infrastructure',
            'confidence': item.get('confidence'),
            'prediction_source': item.get('prediction_source'),
            'mapper_fallback': bool(item.get('mapper_fallback', False)),
            'used_model_fallback': item.get('prediction_source') in {'direct_fallback', 'hard_fallback'},
            'input_tokens': usage.get('input_tokens'),
            'output_tokens': usage.get('output_tokens'),
            'hit_max_new_tokens': bool(usage.get('hit_max_new_tokens', False)),
            'duration_seconds': item.get('duration_seconds'),
            'error': item.get('error'),
        })
    frame = pd.DataFrame(rows)
    valid = frame[frame['is_valid_output']].copy()
    n_total, n_valid = len(frame), len(valid)
    n_failed = n_total - n_valid
    n_infrastructure_failed = int(frame['is_infrastructure_failure'].sum())
    if n_failed:
        failed_ids = frame.loc[~frame['is_valid_output'], 'case_id'].tolist()
        print(
            f'Cảnh báo {model} seed={eval_seed}: {n_failed}/{n_total} output không hợp lệ '
            f'được tính sai. Case: {failed_ids}'
        )

    scored_prediction = frame['scored_prediction']
    strict_correct = int((frame['gold'] == scored_prediction).sum())
    strict_accuracy = strict_correct / n_total if n_total else 0.0
    valid_accuracy = accuracy_score(valid['gold'], valid['prediction']) if n_valid else 0.0
    report = classification_report(
        frame['gold'], scored_prediction, labels=LABELS,
        output_dict=True, zero_division=0,
    ) if n_total else {}
    cm_all = confusion_matrix(
        frame['gold'], scored_prediction, labels=LABELS + [INVALID_OUTPUT_LABEL]
    ) if n_total else np.zeros((len(LABELS) + 1, len(LABELS) + 1), dtype=int)
    cm = cm_all[:len(LABELS), :]
    summary = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'model': model,
        'seed': eval_seed,
        'n_total': n_total,
        'n_success': n_valid,
        'n_failed': n_failed,
        'n_infrastructure_failed': n_infrastructure_failed,
        'invalid_output_rate': n_failed / n_total if n_total else 0.0,
        'coverage': n_valid / n_total if n_total else 0.0,
        'benchmark_valid': (
            n_total == len(inference_cases)
            and n_infrastructure_failed == 0
            and not bool((frame['prediction_source'] == 'hard_fallback').any())
        ),
        'all_outputs_valid': n_valid == n_total,
        'metric_scope': 'all_50_invalid_outputs_count_as_wrong',
        'strict_accuracy_all_50': strict_accuracy,
        'accuracy_successful_only': valid_accuracy,
        'macro_precision': report.get('macro avg', {}).get('precision', 0.0),
        'macro_recall': report.get('macro avg', {}).get('recall', 0.0),
        'macro_f1': report.get('macro avg', {}).get('f1-score', 0.0),
        'weighted_f1': report.get('weighted avg', {}).get('f1-score', 0.0),
        'avg_input_tokens': frame['input_tokens'].mean(),
        'avg_output_tokens': frame['output_tokens'].mean(),
        'avg_duration_seconds': frame['duration_seconds'].mean(),
        'n_hit_max_new_tokens': int(frame['hit_max_new_tokens'].sum()),
        'n_mapper_fallback': int(frame['mapper_fallback'].sum()),
        'n_model_fallback': int(frame['used_model_fallback'].sum()),
        'n_hard_fallback': int((frame['prediction_source'] == 'hard_fallback').sum()),
        'prediction_sources': frame['prediction_source'].value_counts(dropna=False).to_dict(),
    }
    per_label = pd.DataFrame([
        {
            'model': model,
            'seed': eval_seed,
            'label': label,
            'precision': report.get(label, {}).get('precision', 0.0),
            'recall': report.get(label, {}).get('recall', 0.0),
            'f1': report.get(label, {}).get('f1-score', 0.0),
            'support': int(report.get(label, {}).get('support', 0)),
        } for label in LABELS
    ])
    cm_frame = pd.DataFrame(
        cm,
        index=[f'gold_{x}' for x in LABELS],
        columns=[f'pred_{x}' for x in LABELS + [INVALID_OUTPUT_LABEL]],
    )
    return summary, per_label, cm_frame, frame

summaries = []
evaluation_artifacts = {}
for model, seed_results in all_model_results.items():
    evaluation_artifacts[model] = {}
    for eval_seed, results in seed_results.items():
        summary, per_label, cm_frame, case_frame = evaluate_run(model, eval_seed, results)
        summaries.append(summary)
        evaluation_artifacts[model][eval_seed] = {
            'per_label': per_label,
            'confusion_matrix': cm_frame,
            'cases': case_frame,
        }
        print(f'\n=== {model} | seed={eval_seed} ===')
        display(pd.DataFrame([summary]))
        display(cm_frame)

run_metrics = pd.DataFrame(summaries).sort_values(['model', 'seed']).reset_index(drop=True)
aggregate_metrics = run_metrics.groupby('model', as_index=False).agg(
    n_seeds=('seed', 'nunique'),
    accuracy_mean=('strict_accuracy_all_50', 'mean'),
    accuracy_std=('strict_accuracy_all_50', 'std'),
    macro_f1_mean=('macro_f1', 'mean'),
    macro_f1_std=('macro_f1', 'std'),
    invalid_rate_mean=('invalid_output_rate', 'mean'),
    invalid_rate_std=('invalid_output_rate', 'std'),
    avg_output_tokens=('avg_output_tokens', 'mean'),
    avg_duration_seconds=('avg_duration_seconds', 'mean'),
)
aggregate_metrics[['accuracy_std', 'macro_f1_std', 'invalid_rate_std']] = (
    aggregate_metrics[['accuracy_std', 'macro_f1_std', 'invalid_rate_std']].fillna(0.0)
)
leaderboard = aggregate_metrics.sort_values(
    ['accuracy_mean', 'macro_f1_mean'], ascending=False
).reset_index(drop=True)

majority_label = pd.Series(list(gold_by_case.values())).value_counts().idxmax()
majority_accuracy = pd.Series(list(gold_by_case.values())).value_counts().max() / len(gold_by_case)
print(f'Majority baseline: {majority_label} | accuracy={majority_accuracy:.4f}')
print('Per-seed metrics:')
display(run_metrics)
print('Aggregate mean ± std across seeds:')
display(leaderboard)

for model, seed_artifacts in evaluation_artifacts.items():
    slug = slugify(model)
    model_runs = run_metrics[run_metrics['model'] == model]
    model_summary = leaderboard[leaderboard['model'] == model]
    model_runs.to_csv(
        OUTPUT_DIR / f'model_metrics_by_seed_{slug}.csv', index=False, encoding='utf-8-sig'
    )
    model_summary.to_csv(
        OUTPUT_DIR / f'model_metrics_summary_{slug}.csv', index=False, encoding='utf-8-sig'
    )
    for eval_seed, artifacts in seed_artifacts.items():
        suffix = f'{slug}_seed-{eval_seed}'
        artifacts['per_label'].to_csv(
            OUTPUT_DIR / f'metrics_per_label_{suffix}.csv', index=False, encoding='utf-8-sig'
        )
        artifacts['confusion_matrix'].to_csv(
            OUTPUT_DIR / f'confusion_matrix_{suffix}.csv', encoding='utf-8-sig'
        )
        artifacts['cases'].to_csv(
            OUTPUT_DIR / f'case_predictions_{suffix}.csv', index=False, encoding='utf-8-sig'
        )

# ---- Paired comparison với predictions của base trên đúng 50 case ----
base_prediction_name = f'predictions_qwen3-4b_seed-{EVAL_SEED}.json'
base_candidates = []
if os.getenv('ALQAC_BASE_PREDICTIONS_PATH'):
    base_candidates.append(Path(os.environ['ALQAC_BASE_PREDICTIONS_PATH']))
base_candidates += [
    DATA_PATH.parent / base_prediction_name,
    PROJECT_ROOT / base_prediction_name,
    PROJECT_ROOT / 'bge' / 'base' / base_prediction_name,
    Path.cwd() / 'bge' / 'base' / base_prediction_name,
    Path('/root/bge/base') / base_prediction_name,
    Path.cwd() / base_prediction_name,
]
base_prediction_path = next((path.resolve() for path in base_candidates if path.is_file()), None)
if base_prediction_path is None:
    print('Không tìm thấy base predictions; bỏ qua paired comparison. Candidates:', base_candidates)
else:
    base_results = json.loads(base_prediction_path.read_text(encoding='utf-8'))
    pado_results = all_model_results[MODEL][EVAL_SEED]
    paired_rows = []
    for case in inference_cases:
        cid = case['case_id']
        base_prediction = (base_results.get(cid) or {}).get('prediction')
        pado_prediction = (pado_results.get(cid) or {}).get('prediction')
        gold = gold_by_case[cid]
        paired_rows.append({
            'case_id': cid, 'gold': gold,
            'base_prediction': base_prediction, 'pado_prediction': pado_prediction,
            'base_correct': base_prediction == gold,
            'pado_correct': pado_prediction == gold,
            'changed': base_prediction != pado_prediction,
        })
    paired = pd.DataFrame(paired_rows)
    paired_summary = {
        'base_path': str(base_prediction_path),
        'n_cases': len(paired),
        'base_accuracy': float(paired['base_correct'].mean()),
        'pado_accuracy': float(paired['pado_correct'].mean()),
        'accuracy_delta': float(paired['pado_correct'].mean() - paired['base_correct'].mean()),
        'n_changed': int(paired['changed'].sum()),
        'n_improved': int((~paired['base_correct'] & paired['pado_correct']).sum()),
        'n_regressed': int((paired['base_correct'] & ~paired['pado_correct']).sum()),
    }
    transition = pd.crosstab(
        paired['base_prediction'].fillna(INVALID_OUTPUT_LABEL),
        paired['pado_prediction'].fillna(INVALID_OUTPUT_LABEL),
        rownames=['base'], colnames=['pado'], dropna=False,
    )
    paired.to_csv(OUTPUT_DIR / 'paired_cases_vs_base.csv', index=False, encoding='utf-8-sig')
    transition.to_csv(OUTPUT_DIR / 'prediction_transition_vs_base.csv', encoding='utf-8-sig')
    atomic_write_json(OUTPUT_DIR / 'paired_summary_vs_base.json', paired_summary)
    print('Paired comparison vs base:')
    display(pd.DataFrame([paired_summary]))
    display(transition)



=== Qwen3-4B | seed=2026 ===


,benchmark_version,benchmark_protocol,model,seed,n_total,n_success,n_failed,n_infrastructure_failed,invalid_output_rate,coverage,...,macro_f1,weighted_f1,avg_input_tokens,avg_output_tokens,avg_duration_seconds,n_hit_max_new_tokens,n_mapper_fallback,n_model_fallback,n_hard_fallback,prediction_sources
0,v10_legal_pado_thinking_base_matched,legal_pado_thinking_mapper_3vote_derived_verdict,Qwen3-4B,2026,50,50,0,0,0.0,1.0,...,0.267104,0.340128,41858.2,5568.26,15.375,1,50,0,0,{'judge_ensemble': 50}


,pred_A_WIN,pred_B_WIN,pred_PARTIAL_A_WIN,pred_PARTIAL_B_WIN,pred___INVALID_OUTPUT__
gold_A_WIN,4,1,10,1,0
gold_B_WIN,0,2,6,2,0
gold_PARTIAL_A_WIN,3,2,12,2,0
gold_PARTIAL_B_WIN,0,1,4,0,0


Majority baseline: PARTIAL_A_WIN | accuracy=0.3800
Per-seed metrics:


,benchmark_version,benchmark_protocol,model,seed,n_total,n_success,n_failed,n_infrastructure_failed,invalid_output_rate,coverage,...,macro_f1,weighted_f1,avg_input_tokens,avg_output_tokens,avg_duration_seconds,n_hit_max_new_tokens,n_mapper_fallback,n_model_fallback,n_hard_fallback,prediction_sources
0,v10_legal_pado_thinking_base_matched,legal_pado_thinking_mapper_3vote_derived_verdict,Qwen3-4B,2026,50,50,0,0,0.0,1.0,...,0.267104,0.340128,41858.2,5568.26,15.375,1,50,0,0,{'judge_ensemble': 50}


Aggregate mean ± std across seeds:


,model,n_seeds,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,invalid_rate_mean,invalid_rate_std,avg_output_tokens,avg_duration_seconds
0,Qwen3-4B,1,0.36,0.0,0.267104,0.0,0.0,0.0,5568.26,15.375


Không tìm thấy base predictions; bỏ qua paired comparison. Candidates: [PosixPath('/root/predictions_qwen3-4b_seed-2026.json'), PosixPath('/root/predictions_qwen3-4b_seed-2026.json'), PosixPath('/root/bge/base/predictions_qwen3-4b_seed-2026.json'), PosixPath('/root/bge/base/predictions_qwen3-4b_seed-2026.json'), PosixPath('/root/bge/base/predictions_qwen3-4b_seed-2026.json'), PosixPath('/root/predictions_qwen3-4b_seed-2026.json')]


In [9]:
for model, seed_results in all_model_results.items():
    for eval_seed, results in seed_results.items():
        submission = []
        for case in inference_cases:
            item = results.get(case['case_id'], {})
            if item.get('prediction') not in LABELS:
                continue
            submission.append({
                'case_id': case['case_id'],
                'prediction': item['prediction'],
                # case_evidence = id cac segment evidence THAT cua case (agent_v4).
                'case_evidence': evidence_ids_by_case.get(case['case_id'], []),
                'law_evidence': [
                    {'law_id': law['law_id'], 'aid': int(law['aid'])}
                    for law in item.get('applied_laws', [])
                ],
            })
        expected_ids = {case['case_id'] for case in inference_cases}
        submission_ids = [row['case_id'] for row in submission]
        if len(submission) != len(inference_cases) or set(submission_ids) != expected_ids:
            raise RuntimeError(
                f'Submission coverage failed: rows={len(submission)}, unique={len(set(submission_ids))}'
            )
        path = OUTPUT_DIR / f'submission_{slugify(model)}_seed-{eval_seed}.json'
        atomic_write_json(path, submission)
        n_failed = len(inference_cases) - len(submission)
        print(
            model,
            '| seed:', eval_seed,
            '| evaluated:', len(inference_cases), '/ 50',
            '| invalid counted wrong:', n_failed,
            '| valid submission rows:', len(submission), '/ 50',
            '|', path,
        )

print('Outputs:', OUTPUT_DIR.resolve())


Qwen3-4B | seed: 2026 | evaluated: 50 / 50 | invalid counted wrong: 0 | valid submission rows: 50 / 50 | outputs_alqac_e2e/v10_legal_pado_thinking_base_matched/submission_qwen3-4b_seed-2026.json
Outputs: /root/outputs_alqac_e2e/v10_legal_pado_thinking_base_matched


In [10]:
run_metrics[['strict_accuracy_all_50', 'macro_precision', 'macro_recall', 'macro_f1']]

,strict_accuracy_all_50,macro_precision,macro_recall,macro_f1
0,0.36,0.31994,0.270395,0.267104


In [11]:
# Dong Modal app: giai phong container GPU remote sau khi chay xong toan bo pipeline.
close_modal()
print('Da dong Modal app.')


(EngineCore pid=486) INFO 08-11 15:19:35 [core.py:1214] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=486) INFO 08-11 15:19:35 [core.py:1333] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=486) INFO 08-11 15:19:35 [core.py:1364] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=486) INFO 08-11 15:19:35 [core.py:1227] [shutdown] EngineCore: exiting busy loop


(EngineCore pid=486) Process EngineCore:
(EngineCore pid=486) Traceback (most recent call last):
(EngineCore pid=486)   File "/usr/local/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1224, in run_engine_core
(EngineCore pid=486)     engine_core.run_busy_loop()
(EngineCore pid=486)   File "/usr/local/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1267, in run_busy_loop
(EngineCore pid=486)     raise SystemExit
(EngineCore pid=486) SystemExit
(EngineCore pid=486) 
(EngineCore pid=486) During handling of the above exception, another exception occurred:
(EngineCore pid=486) 
(EngineCore pid=486) Traceback (most recent call last):
(EngineCore pid=486)   File "/usr/local/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=486)     self.run()
(EngineCore pid=486)   File "/usr/local/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=486)     self._target(*self._args, **self._kwargs)
(EngineCore pid=486)   File "

Da giai phong GPU local.
Da dong Modal app.
